# Create datasets

This notebook documents the preparation and construction of the datasets used in this thesis. The final constructed datasets are provided under "data/biomass_datasets/" and can be used directly. This notebook is included to make their construction reproducible.

The pipeline below assumes that the underlying data have already been generated using the same experimental environments and configurations described in this project. Instructions for generating these data are provided in the project README. Given the generated data, this notebook describes the steps required to process and construct the final datasets used throughout the project.


## Table of contents

- [1. macro_compostion.pkl](#1.-macro_compostion.pkl)
- [2. internal_biomass.pkl](#2.-internal_biomass.pkl)
- [3. Training and test sets for ML](#3.-Machine-Learning-Data)


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path(os.getcwd())

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FILES = ROOT / "files"
DATA = ROOT / "data"
PHPP = DATA / "PHPP_samples"

sys.path.append(str(ROOT))

from scripts.EDA.data_prep import build_substrate_df, fix_col_names, merge_samples, get_samples

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## 1. macro_composition.pkl

### Make DataFrame for each environment label (substrate)

*Carbon*

In [ ]:
glc = build_substrate_df(sample_path=PHPP/"glc/flux",
                         shadow_path=PHPP/"glc/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'glc',
                         metabolites=['glc__D_e', 'o2_e'])


ac = build_substrate_df(sample_path=PHPP/"ac/flux",
                         shadow_path=PHPP/"ac/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'ac',
                         metabolites=['ac_e', 'o2_e'])


lcts = build_substrate_df(sample_path=PHPP/"lcts/flux",
                         shadow_path=PHPP/"lcts/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'lcts',
                         metabolites=['lcts_e', 'o2_e'])


glyc = build_substrate_df(sample_path=PHPP/"glyc/flux",
                         shadow_path=PHPP/"glyc/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'glyc',
                         metabolites=['glyc_e', 'o2_e'])

succ = build_substrate_df(sample_path=PHPP/"succ/flux",
                         shadow_path=PHPP/"succ/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'succ',
                         metabolites=['succ_e', 'o2_e'])

*Nitrogen*

In [ ]:
arg_glc = build_substrate_df(sample_path=PHPP/"arg_glc/flux",
                         shadow_path=PHPP/"arg_glc/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'arg_glc',
                         metabolites=['arg__L_e', 'o2_e'])

arg_ac = build_substrate_df(sample_path=PHPP/"arg_ac/flux",
                         shadow_path=PHPP/"arg_ac/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'arg_ac',
                         metabolites=['arg__L_e', 'o2_e'])

arg_glyc = build_substrate_df(sample_path=PHPP/"arg_glyc/flux",
                         shadow_path=PHPP/"arg_glyc/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'arg_glyc',
                         metabolites=['arg__L_e', 'o2_e'])



nh4_glc = build_substrate_df(sample_path=PHPP/"nh4_glc/flux",
                         shadow_path=PHPP/"nh4_glc/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'nh4_glc',
                         metabolites=['nh4_e', 'o2_e'])

nh4_ac = build_substrate_df(sample_path=PHPP/"nh4_ac/flux",
                         shadow_path=PHPP/"nh4_ac/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'nh4_ac',
                         metabolites=['nh4_e', 'o2_e'])



no3_glc = build_substrate_df(sample_path=PHPP/"no3_glc/flux",
                         shadow_path=PHPP/"no3_glc/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'no3_glc',
                         metabolites=['no3_e', 'o2_e'])

no3_glyc = build_substrate_df(sample_path=PHPP/"no3_glyc/flux",
                         shadow_path=PHPP/"no3_glyc/shadow",
                         template_path=FILES/"macro_template.csv",
                         substrate = 'no3_glyc',
                         metabolites=['no3_e', 'o2_e'])

*MERGE*

In [ ]:
macro_data = pd.concat([glc, ac, lcts, glyc, succ, arg_glc, arg_ac, arg_glyc, nh4_glc, nh4_ac, no3_glc, no3_glyc],
                     ignore_index=True)

Uncomment below and run cell to save macro_composition.pkl

In [ ]:
# macro_data.to_pickle(macro_composition.pkl')

## 2. internal_biomass.pkl

* bm_rxns_template.csv

In [ ]:
glc = get_samples(PHPP/"glc/flux")
ac = get_samples(PHPP/"ac/flux")
lcts = get_samples(PHPP/"lcts/flux")
glyc = get_samples(PHPP/"glyc/flux")
succ = get_samples(PHPP/"succ/flux")

arg_glc = get_samples(PHPP/"arg_glc/flux")
arg_ac = get_samples(PHPP/"arg_ac/flux")
arg_glyc = get_samples(PHPP/"arg_glyc/flux")

nh4_glc = get_samples(PHPP/"nh4_glc/flux")
nh4_ac = get_samples(PHPP/"nh4_ac/flux")

no3_glc = get_samples(PHPP/"no3_glc/flux")
no3_glyc = get_samples(PHPP/"no3_glyc/flux")

In [ ]:
samples = glc+ac+lcts+glyc+succ+arg_glc+arg_ac+arg_glyc+nh4_glc+nh4_ac+no3_glc+no3_glyc

Uncomment below and run cell to save internal_biomass.pkl

In [ ]:
# internal_biomass = merge_samples(samples, internal_biomass.pkl")

## 3. Machine Learning Data

* split train/test
* make features

In [3]:
from sklearn.linear_model import LinearRegression

**SPLIT**

In [17]:
data = pd.read_pickle(DATA/'biomass_datasets/macro_composition.pkl')

In [27]:
train_set, test_set = train_test_split(data.index, test_size=0.2, stratify=data['source'], random_state=42)

train_data = data.loc[train_set]
test_data = data.loc[test_set]

### Feature matrix

In [28]:
ex_cols = [col for col in data.columns if col.startswith('EX_')]
bm_cols = [col for col in data.columns if 'mass' in col]
rna_cols = data['mRNA_biomass_to_biomass'] + data['rRNA_biomass_to_biomass'] + data['tRNA_biomass_to_biomass']

**Compute ratios**

In [31]:
ratio_train = train_data[bm_cols]
ratio_test = test_data[bm_cols]

########## TRAIN ###########################
ratio_train.drop(columns=['dummy_protein_to_mass', 'peptidoglycan_biomass_to_biomass', 'constituent_biomass_to_biomass'], inplace=True)
ratio_train.drop(columns=[col for col in ratio_train.columns if 'RNA' in col and 'ncRNA' not in col], inplace=True)
ratio_train.insert(1, 'RNA_biomass_to_biomass', rna_cols)

########## TEST ####################################
ratio_test.drop(columns=['dummy_protein_to_mass', 'peptidoglycan_biomass_to_biomass', 'constituent_biomass_to_biomass'], inplace=True)
ratio_test.drop(columns=[col for col in ratio_test.columns if 'RNA' in col and 'ncRNA' not in col], inplace=True)
ratio_test.insert(1, 'RNA_biomass_to_biomass', rna_cols)


In [32]:
biomass_train = ratio_train.copy()
biomass_test = ratio_test.copy()

In [38]:
ratio_train['ncRNA/RNA'] = ratio_train['ncRNA_biomass_to_biomass']/ratio_train['RNA_biomass_to_biomass']
ratio_train['protein/DNA'] = ratio_train['protein_biomass_to_biomass']/ratio_train['DNA_biomass_to_biomass']
ratio_train['RNA/DNA'] = ratio_train['RNA_biomass_to_biomass']/ratio_train['DNA_biomass_to_biomass']
ratio_train['ncRNA/DNA'] = ratio_train['ncRNA_biomass_to_biomass']/ratio_train['DNA_biomass_to_biomass']


ratio_test['ncRNA/RNA'] = ratio_test['ncRNA_biomass_to_biomass']/ratio_test['RNA_biomass_to_biomass']
ratio_test['protein/DNA'] = ratio_test['protein_biomass_to_biomass']/ratio_test['DNA_biomass_to_biomass']
ratio_test['RNA/DNA'] = ratio_test['RNA_biomass_to_biomass']/ratio_test['DNA_biomass_to_biomass']
ratio_test['ncRNA/DNA'] = ratio_test['ncRNA_biomass_to_biomass']/ratio_test['DNA_biomass_to_biomass']

**LOG-TRANSFORM**

In [40]:
log_train = ratio_train.copy()
log_train = np.log(log_train)

log_test = ratio_test.copy()
log_test = np.log(log_test)

In [42]:
train_features = log_train.drop(columns='lipid_biomass_to_biomass')
test_features = log_test.drop(columns='lipid_biomass_to_biomass')

**L2-NORM**

In [47]:
l2_train = train_features.iloc[:, :5]
l2_test = test_features.iloc[:, :5]

In [49]:
from sklearn.preprocessing import normalize

train_transformed =  pd.DataFrame(
    normalize(l2_train.values, norm='l2', axis=1),
    index=l2_train.index,
    columns=l2_train.columns
)

In [52]:
test_transformed = pd.DataFrame(
    normalize(l2_test.values, norm='l2', axis=1),
    index=l2_test.index,
    columns=l2_test.columns
)

**Feature DataFrames**

In [56]:
train_df = train_transformed.copy()
test_df = test_transformed.copy()

train_df[['ncRNA/RNA', 'protein/DNA','RNA/DNA','ncRNA/DNA', 'biomass_dilution']] = log_train[['ncRNA/RNA', 'protein/DNA','RNA/DNA','ncRNA/DNA', 'biomass_dilution']]
test_df[['ncRNA/RNA', 'protein/DNA','RNA/DNA','ncRNA/DNA', 'biomass_dilution']] = log_test[['ncRNA/RNA', 'protein/DNA','RNA/DNA','ncRNA/DNA', 'biomass_dilution']]

In [68]:
train_df[ex_cols] = train_data[ex_cols]
train_df[['source', 'outlier', 'o2_e_shadow', 'source_shadow']] = train_data[['source', 'outlier', 'o2_e_shadow', 'source_shadow']]

test_df[ex_cols] = test_data[ex_cols]
test_df[['source', 'outlier', 'o2_e_shadow', 'source_shadow']] = test_data[['source', 'outlier', 'o2_e_shadow', 'source_shadow']]


Uncomment the code and run the cell below to save files

In [ ]:
# train_df.to_pickle('L2_TRAIN.pkl')
# test_df.to_pickle('L2_IID_TEST.pkl')